In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download  
2. Open a terminal and run:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also use:

```bash
curl http://localhost:11434
```

If needed, you can install the Python client with:

```bash
pip install ollama
```


In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t have a FAQ entry for running **Ollama/Olama locally** in the provided context.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Absolutely — in most cases, yes, you can join even if you’ve just discovered it.\n\nA few things to check:\n- **Whether enrollment is still open**\n- **Any prerequisites** or required background\n- **Whether there’s a registration deadline**\n- **If the course has a waiting list**\n\nIf you want, I can help you figure out the next step. Just send me:\n- the **course name or link**, and\n- where you’re seeing it offered (school, platform, event, etc.)\n\nThen I can help you determine if you can still join and what to do next.'

In [8]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [10]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [11]:
len(response.output)

3

In [12]:
call = response.output[0]

In [13]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration FAQ"}', call_id='call_8nWmBzb1jCUXZy5mAOEm1BLa', name='search', type='function_call', id='fc_0c36c8e0a721168c006a280d2ca13c819f9e1d0a79f4a51ebb', namespace=None, status='completed')

In [14]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [15]:
len(response.output)

2

In [16]:
call = response.output[0]

In [17]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join FAQ"}', call_id='call_XxArV2c4vIyneBMv7XdFtS0U', name='search', type='function_call', id='fc_096d8c07923172af006a280d36060c81a28621ea63749a111b', namespace=None, status='completed')

In [18]:
import json
args = json.loads(call.arguments)

In [19]:
results = search(**args)

In [20]:
result_json = json.dumps(results,indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course

In [21]:
function_call_output={
    'type': 'function_call_output',
    'call_id': call.call_id,
    'output': result_json
}

In [22]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [23]:
messages.append(call)

In [24]:
messages.append(function_call_output)

In [25]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [26]:
print(response.output_text)

In [27]:
usage = response.usage
usage.input_tokens, usage.output_tokens, usage.total_tokens

(748, 29, 777)

In [38]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [39]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [40]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [43]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course enroll late registration discovered the course can I join"}', call_id='call_v9GLtMFqVadv64fK5K9ATQsI', name='search', type='function_call', id='fc_0c34ea2518cd30e8006a2818121a6481a094fa974130772555', namespace=None, status='completed')]

In [44]:
len(response.output)

1

In [48]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"join the course enroll late registration discovered the course can I join"}', call_id='call_v9GLtMFqVadv64fK5K9ATQsI', name='search', type='function_call', id='fc_0c34ea2518cd30e8006a2818121a6481a094fa974130772555', namespace=None, status='completed')

In [49]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late registration"}
function_call: search {"query":"new student join course enrollment FAQ discovered course"}
function_call: search {"query":"course access after start date can I join late FAQ"}


In [36]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join the course enroll late registration discovered the course can I join"}


In [50]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course. 

If you want a certificate, make sure to submit your project while submissions are still open. If the live cohort has already finished, you can still follow along, but certificates aren’t available in self-paced mode.

If you want, I can also help with the next steps for joining or explain the certificate requirements. Is there anything else you’d like to explore?


In [51]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [52]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama run locally install local model FAQ"}
iteration #2...
function_call: search {"query":"ollama serve localhost 11434 run llama3 python client FAQ"}
iteration #3...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model, starts it locally, and opens a chat-style prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a JSON response.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   p

'To run Ollama locally:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model, starts it locally, and opens a chat-style prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a JSON response.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error, you may need to restart the server with:\n```bash\nollama serve\n```\n\nIf you want, I can also show you 

In [53]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered the course can I still join"}
function_call: search {"query":"course enrollment open late join discovered course"}
function_call: search {"query":"can I still join the course if I discovered it late FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, just make sure you submit your project while submissions are still open. If you’re joining later, you can still follow along and learn, but certificate eligibility depends on the live course timeline.

If you want, I can also help with what you need to do next to get started.


'Yes — you can still join the course.\n\nIf you want a certificate, just make sure you submit your project while submissions are still open. If you’re joining later, you can still follow along and learn, but certificate eligibility depends on the live course timeline.\n\nIf you want, I can also help with what you need to do next to get started.'

In [54]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered late enrollment can I join course late FAQ"}
iteration #2...
function_call: search {"query":"certificate project while accepting submissions peer review live cohort self-paced FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want to earn a certificate, make sure you submit your project while submissions are still open. Certificates aren’t available for self-paced participation after the course’s live cohort ends.

If you’d like, I can also explain how registration, homework submission, and certificates work. Is there anything else you want to explore?


'Yes — you can still join the course.\n\nIf you want to earn a certificate, make sure you submit your project while submissions are still open. Certificates aren’t available for self-paced participation after the course’s live cohort ends.\n\nIf you’d like, I can also explain how registration, homework submission, and certificates work. Is there anything else you want to explore?'

In [55]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit opening chess queen's gambit definition"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening sacrifice of the c-pawn explanation"}
iteration #3...
ASSISTANT:
A **Queen’s Gambit** is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a gambit to try to pull Black’s **d-pawn** away from the center and gain better control of the board.

### Why it’s called a gambit
A **gambit** is an opening where one side offers a pawn or another small material sacrifice for a strategic advantage, like:
- stronger center control
- faster development
- better piece activity

### Main idea
White wants to:
- challenge Black’s central pawn on **d5**
- open lines for pieces
- create pressure in the center

### Common responses
Black can accept or decline the gambit:
- **Queen’s Gambit Accepted**: `1.d4 d5 2.c4 dxc4`
- **Queen’s Gambit Declined**: `1.d4 d5 2.c4 e6`
- There are other def

'A **Queen’s Gambit** is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a gambit to try to pull Black’s **d-pawn** away from the center and gain better control of the board.\n\n### Why it’s called a gambit\nA **gambit** is an opening where one side offers a pawn or another small material sacrifice for a strategic advantage, like:\n- stronger center control\n- faster development\n- better piece activity\n\n### Main idea\nWhite wants to:\n- challenge Black’s central pawn on **d5**\n- open lines for pieces\n- create pressure in the center\n\n### Common responses\nBlack can accept or decline the gambit:\n- **Queen’s Gambit Accepted**: `1.d4 d5 2.c4 dxc4`\n- **Queen’s Gambit Declined**: `1.d4 d5 2.c4 e6`\n- There are other defenses too, like the **Slav Defense**\n\nIf you want, I can also explain the **Queen’s Gambit Accepted vs Declined** in a very simple way, or show a few sample moves.'

In [56]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening queen gambit"}
iteration #3...
ASSISTANT:
I couldn’t find a course-related FAQ entry for “queen gambit,” so I can’t answer that from the course database.

If you meant something from the course materials, could you rephrase it with the exact term used there? Otherwise, feel free to ask about another course topic or logistics area you want to explore.


'I couldn’t find a course-related FAQ entry for “queen gambit,” so I can’t answer that from the course database.\n\nIf you meant something from the course materials, could you rephrase it with the exact term used there? Otherwise, feel free to ask about another course topic or logistics area you want to explore.'

In [57]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [58]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [59]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [60]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [61]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [62]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [63]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)